# 02 — Which records will we learn from, tune on, and keep for the final test?

Our goal is to compare tag-suggestion methods fairly. This notebook examines historical
labels and the **saved benchmark split**. It does not train a model or choose a new split.

| Dataset part | What we will use it for later |
|---|---|
| Training | Fit classifiers and text features using historical tags. |
| Validation | Choose model settings, prompts, and thresholds. The rubric pilot is drawn from here. |
| Final test | Evaluate chosen methods once settings and independently reviewed tags are fixed. |

There is also a **40-record rubric pilot**: a subset of validation used to try out the
human labeling rules. It is not a fourth dataset part. A **rubric** is the written guide
for deciding which topic tags apply. Historical tags can be wrong; new review forms start blank.

The first table shows whether split membership is **locked** or still awaiting group review.
Locked membership means later notebooks use these saved assignments instead of drawing new splits.
This follows [the project plan, sections 7.3 and 8.2](../docs/adr/0001-capstone-backoffice-ml-vs-llm.md).

**Before running:** use the [supported kernel](../docs/notebook-setup.md), complete
[Notebook 01](01_data_snapshot_and_quality.ipynb), and restore the matching five-file
[private benchmark bundle](../docs/data-preparation.md#restore-the-private-benchmark)
under `data/benchmark/`. This notebook validates and loads existing artifacts.
It cannot create them from the snapshot alone.
Run this notebook top to bottom. Tables show aggregate counts; private review files stay
under `data/`. Clear all outputs before committing.

**Where to review DR-34:** after running all cells, check the **Benchmark fact** table
in section **1** for **Split membership: locked** and **Reviewed event types: 29**.
Then check the table in section **6** for **2,626 training**, **659 validation**, and
**149 final-test** records. You do not need to repeat the CSV review.

These assignments are fixed for the model comparison. Label review can continue.
Changing which demonstrations belong to each split requires a new recorded split version.
The files remain editable; the loader checks that the recorded membership has not changed.


**Working approach:** explore data and models in notebooks first. Shared helpers validate inputs and display tables; the analysis is visible in the cells. Service implementation follows model evaluation and selection.


## 1. Load the same verified snapshot

We verify the same snapshot as Notebook 01, then apply the same rule: a title must
have at least two words to enter EDA, training, validation, or final test.
For example, `"Peace"` is excluded and `"Peace now"` is kept. Whitespace separates
words; punctuation alone does not count. The snapshot and its historical tags are unchanged. A failed validation
stops the analysis. The table helper only controls presentation; it does not change records.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path


root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the DemoLogic repository or notebooks/ folder")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from notebooks.tools import show_table


from demo_logic.notebook_support import tagging_eligible_rows, title_word_count
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags


from demo_logic.benchmark import load_benchmark

snapshot_rows, source_manifest = load_snapshot(
    root / "data/dataset.jsonl", root / "data/manifest.json"
)
TOPIC_TAGS = topic_tags(source_manifest["taxonomy_version"])
rows = tagging_eligible_rows(snapshot_rows)
excluded_title_rows = [row for row in snapshot_rows if title_word_count(row["title"]) < 2]
if not rows:
    raise ValueError(
        "No records have at least two title words. Inspect the snapshot before loading the saved split."
    )
benchmark_dir = root / "data" / "benchmark"
benchmark = load_benchmark(benchmark_dir, snapshot_rows, source_manifest["content_sha256"])
split_manifest = benchmark.manifest
groups = benchmark.groups
pilot_ids = benchmark.pilot_ids
split_membership = "locked" if split_manifest["status"] == "frozen" else "awaiting group review"

show_table(
    ["Benchmark fact", "Value"],
    [
        ("Loaded taxonomy", source_manifest["taxonomy_version"]),
        ("Split membership", split_membership),
        (
            "Membership locked at (UTC)",
            split_manifest.get("freeze", {}).get("frozen_at", "Not locked"),
        ),
        ("Reviewed event types", benchmark.group_review_count),
        ("Verified snapshot", len(snapshot_rows)),
        ("Excluded: fewer than two title words", len(excluded_title_rows)),
        ("Eligible for EDA and splitting", len(rows)),
    ],
)

## 2. What do the historical tags look like?

**EDA (exploratory data analysis)** means describing the data before modeling it.
Count records carrying each tag, including tags with zero records. A small count warns
that a tag may be difficult to learn or evaluate. Counts are not accuracy scores.

A record can have two tags. The second table shows which pairs occur together most often:
for example, an event tagged both `mobility` and `environment` adds one to that pair's count.
Frequent combinations can reflect real overlap or inconsistent labeling; review decides.
Prepared data contains no untagged records because preparation excludes them.

In [ ]:
tag_counts = Counter(tag for row in rows for tag in row["tag_slugs"])
show_table(
    ["Historical tag", "Records", "% of all records"],
    [
        (tag, tag_counts[tag], f"{100 * tag_counts[tag] / len(rows):.1f}%")
        for tag in sorted(TOPIC_TAGS, key=lambda tag: (tag_counts[tag], tag))
    ],
)

### Which tags occur together?

First count records with one or two tags. Then show the five most common pairs. Each pair count refers to records carrying both tags, not the number of tags or groups. Use frequent overlaps to identify questions for the human labeling rubric.


In [ ]:
tag_cardinality = Counter(len(row["tag_slugs"]) for row in rows)
show_table(["Tags on a record", "Records"], [(count, tag_cardinality[count]) for count in (1, 2)])
cooccurrence = Counter(
    tuple(sorted(row["tag_slugs"])) for row in rows if len(row["tag_slugs"]) == 2
)
show_table(
    ["Tag 1", "Tag 2", "Records with both tags"],
    [(tags[0], tags[1], count) for tags, count in cooccurrence.most_common(5)],
)

## 3. What is a group, and why keep it together?

A group can contain several source records that may describe editions of the same
demonstration. For example, two weekly editions of **“Sichere Radwege für Berlin”**
should stay together. Otherwise a model could learn one edition and appear to do well
on an almost identical test record.

The saved manifest records which records were grouped in the reviewed candidate.
The 29 rows in `group-review.csv` are **29 group-review entries**, not the entire
dataset and not necessarily 29 single source records. Each entry was reviewed
separately. DR-34 verified those decisions and locked the existing assignments.


### Count an event type once (DR-40)

The main comparison uses **one original record per saved event group**: 732 training,
188 validation and 123 final-test representatives. The original 2,626 / 659 / 149
occurrences remain in the snapshot and locked split. We choose the earliest date,
then the lowest event ID if dates tie. In the 40 pilot groups, we keep the record
already selected for review. Neither tags nor model scores choose a record.

We use the selected record's original title and historical tags. If a group has
different title variants or historical tag sets, we retain that fact as a limitation;
we do not combine labels or replace the title with the group's display name.
There are 42 training and four validation groups with conflicting historical tags.
The private selection manifest preserves all member IDs, and every report records
its fingerprint. Earlier occurrence-based reports remain historical experiments
and must not be compared directly with these new results.

The EDA tables below describe the full source occurrences. Notebooks 03 and 04 apply the representative selection for modeling.


## 4. How were the three dataset parts chosen?

Imagine we have **500 demonstration records**. We reserve about 150 for the final
test, take 20% of the remainder for validation, and use the rest for training:
`150 final + 70 validation + 280 training = 500`.

| Step | What we do | Records assigned | Records still available |
|---|---|---:|---:|
| Start | Begin with all records | — | 500 |
| 1 | Set aside the final test | 150 final test | 350 |
| 2 | Take 20% of those remaining 350 for validation: `350 × 0.20 = 70` | 70 validation | 280 |
| 3 | Use all remaining records for training | 280 training | 0 |

**Check the total: `150 + 70 + 280 = 500`.** Validation is 20% of the records left
*after* reserving the final test, not 20% of the original 500.

Next, we select **40 of the 70 validation records** for the human labeling pilot.
Those 40 still belong to validation. We have not added another 40 records or removed
them from validation. The other 30 validation records are simply not part of the pilot.
The pilot needs at least 40 distinct validation groups because it takes one record per group.

**Why might the actual counts differ?** We move related events together as whole groups.
For example, if the final test has 148 records and the next group contains 4, adding it
would exceed the target of 150. We skip that group and try other groups. If none fits,
the final test stays at 148. Final test must have at least 100 records; validation can
also fall short of its target. Training receives everything left over.

### What does the saved candidate record?

The next cell displays the saved seed, cutoff, grouping policy, and title-eligibility
history. It does not select records using today's date. The cutoff means that every
record in a final-test group must fall on or before that saved date. The group-size
limit also applies to final-test groups.

The current candidate was originally selected with grouping policy v1. The two-word
title rule was applied afterwards by removing ineligible assignments while keeping
the remaining records in their original splits. That history explains why the saved
final set contains 149 records rather than the original target of 150.

Loading this bundle recovers the exact saved assignments. Running the current split
selector on the already-filtered input is a different operation: it can change
membership even with the same seed and cutoff. Preserve the reviewed bundle and
record any deliberate replacement in the private daily audit.

The date limit does not make this a chronological holdout, and group selection does
not guarantee equal tag proportions. Review these limitations before freezing the split.


In [ ]:
show_table(
    ["Saved selection setting", "Value"],
    [
        ("Selection seed", split_manifest["selection_seed"]),
        ("Target final-test records", split_manifest["final_target"]),
        (
            "Validation share of remaining records",
            f"{100 * split_manifest['validation_fraction']:.0f}%",
        ),
        ("Largest group allowed in final test", split_manifest["max_final_group_size"]),
        ("Latest date allowed in final test", split_manifest["latest_final_date"]),
        ("Grouping policy", split_manifest["grouping_policy"]),
        (
            "Title eligibility history",
            split_manifest["title_eligibility"].get("application", "Not recorded"),
        ),
        ("Split membership", split_membership),
    ],
)

## 5. Which groups need human inspection?

The first table counts groups and records separately. A group of four events counts
as **one group** and **four records**. “Title pairs” counts comparisons that matched,
so it is not the same as the number of groups.

Different historical tags within a group can mean changed demands, a bad merge, or
inconsistent old labels. They are review flags, not confirmed label errors. The saved
`group-review.csv` includes varied titles, dates beyond the selected cutoff, and samples of large or
variously tagged groups; it is not an exhaustive review of every possible mistake.

In [ ]:
varied_tags = sum(
    len({tuple(sorted(row["tag_slugs"])) for row in members}) > 1 for members in groups.values()
)
show_table(
    ["Grouping observation", "Count", "Meaning"],
    [
        ("All groups", len(groups), "Families kept together when splitting"),
        (
            "Groups with multiple records",
            sum(len(members) > 1 for members in groups.values()),
            "Recurring or similar events",
        ),
        ("Largest group", max(map(len, groups.values())), "Number of records in one family"),
        (
            "Title pairs matched during original selection",
            split_manifest["similar_title_pair_count"],
            "Historical count from the saved manifest; not recomputed here",
        ),
        (
            "Groups with differing historical tags",
            varied_tags,
            "Check merges and label consistency",
        ),
    ],
)

## 6. How many records were assigned to each dataset part?

This table shows the saved split for the model comparison. Its membership is locked after DR-34 review: records stay in their assigned partition while label review can continue. These counts describe the data, not model performance.

In [ ]:
split_roles = {
    "training": "Fit classifiers and text features",
    "validation": "Choose settings; also supplies the rubric pilot",
    "final_test": "Final comparison after settings and reference labels are fixed",
}
show_table(
    ["Dataset part", "Records", "Groups", "Later purpose"],
    [
        (name, split_manifest["split_counts"][name], split_manifest["group_counts"][name], purpose)
        for name, purpose in split_roles.items()
    ],
)

## 7. Will we have enough examples of each tag?

The table counts historical tag assignments within each part. Final-test numbers are
only a preliminary audit of sample coverage: the independently reviewed tags may differ.
A zero or small count means weak evidence for that tag, not poor model performance.
Record that limitation; do not choose a new split based on future test results.

In [ ]:
show_table(
    ["Historical tag", "Training records", "Validation records", "Final-test records"],
    [
        (tag, *[split_manifest["tag_counts"].get(name, {}).get(tag, 0) for name in split_roles])
        for tag in sorted(TOPIC_TAGS)
    ],
)

## 8. What is the 40-record pilot for?

The pilot is for practicing and improving the [human tagging rubric](../docs/tagging-rubric.md).
It is a subset of validation, not an extra split or a final score for a model.

The original selection chose one record from each of 30 validation groups in a repeatable random order,
then one from each of 10 further groups. This notebook reads that saved selection. The last 10 favor health/care titles and
historically two-tagged records, because those may reveal unclear rubric boundaries.
Other groups fill remaining places when too few targeted examples exist.

Existing tags help choose examples but are hidden in the review CSV. Keep the private
selection-reason file separate from that form. A targeted pilot is not a representative
sample from which to estimate the error rate of all historical labels.

In [ ]:
assignment_by_id = {item["external_id"]: item for item in split_manifest["assignments"]}
show_table(
    ["Pilot property", "Result"],
    [
        ("Records in pilot queue", len(pilot_ids)),
        ("Records with nonempty reviewed_tags", benchmark.pilot_tagged_count),
        ("Source dataset part", "Validation"),
        (
            "Distinct groups represented",
            len({assignment_by_id[event_id]["group_id"] for event_id in pilot_ids}),
        ),
        ("Purpose", "Clarify and freeze the tagging rubric"),
    ],
)

## 9. What has the benchmark loader verified?

The loader in section 1 checks the bundle before any results are displayed:

- Every eligible record is assigned exactly once to a recognised split.
- Each saved group stays in one split. Final groups respect the saved date and size limits.
- Row, group, and tag totals calculated from the assignments match the manifest.
- The final-review IDs match the final-test assignments exactly.
- The pilot contains 40 validation records from different groups, and agrees with its selection IDs, snapshot fingerprint, and seed.
- Group-review IDs, member counts, and date ranges match the saved groups.

A mismatch stops the notebook with instructions to preserve completed reviews and
reconcile the files. These checks establish file consistency, not tag correctness
or approval of the grouping decisions. A nonempty tag field is not proof of a
completed or correct review.


In [ ]:
print("The saved benchmark bundle passed the consistency checks described above.")
print("Split membership:", split_membership)
print("Locked membership fixes which records belong to each split. Label review can continue.")

## 10. Locate the active private benchmark files

The active benchmark is under `data/benchmark/`. The loader has already
checked that all five files exist and agree with the snapshot and saved assignments.
The table below identifies their purposes. This notebook does not write review forms.
Completed reviews stay in these files; the private daily audit records review progress.

The notebook displays **locked** when the manifest records `status: frozen`. The loader
verifies the saved membership fingerprint and reviewed keep-together decisions.
This locks split membership only; it does not approve or lock reference labels.


In [ ]:
file_purposes = {
    "manifest": "Saved assignments, settings, and snapshot fingerprint",
    "group_review": "Group decisions and any preferred-title choices",
    "rubric_pilot": "Pilot tags, uncertainty, source links, and notes",
    "pilot_selection": "Private pilot sampling reasons and IDs",
    "final_review": "Final reference labels, filled after the rubric is frozen",
}
show_table(
    ["Private file", "What it provides"],
    [
        (str(benchmark.paths[name].relative_to(root)), purpose)
        for name, purpose in file_purposes.items()
    ],
)

## 11. What happens after this notebook?

**Group review:** `data/benchmark/group-review.csv` contains the priority group-review
entries. Each entry may cover multiple source records; its `row_count` tells you how
many. The `decision` column records the review, but editing that CSV does not change
the saved assignments. DR-34 verified these decisions and locked the saved membership. The loader checks that the lock still matches the files.
Each entry is one event type that may have several dated occurrences. The 29
reviewed types remain separate from one another. `preferred_title` is a readable
label for the type, even if it has only one title variant. The loader checks the
group ID, count, and dates, not the wording of this label. Each occurrence keeps
its own original title for model input.

**Rubric pilot:** Review the 40 validation records in
`data/benchmark/rubric-pilot-queue.csv`. `reviewed_tags` contains the chosen tag slugs.
`uncertainty` records doubt about those tags: lower values mean more confidence.
Keep any source links and notes with the private review. Use the pilot to resolve
questions in `docs/tagging-rubric.md`, then record the agreed rubric version and date.

**Final test:** `data/benchmark/final-review-queue.csv` holds the final-test records. Label its
records only after freezing the rubric, without looking at model suggestions.
Those human labels will be the answer key for the final comparison.

**Next:** use this checked benchmark in Notebook 03, where you can read and run each baseline rule. Continue model experiments in later notebooks before building a service. Never regenerate blank review forms
over completed work. Record split and rubric decisions in the private daily audit.


## 12. DR-40: What will the models actually use?

The earlier tables count all source occurrences. This table counts one representative per saved group. Check 732 training, 188 validation and 123 final event types. All 40 preselected pilot IDs remain. The helper chooses IDs from dates and group membership only; it does not read final reference labels. Notebooks 03 and 04 save this exact mapping privately and include its fingerprint in their reports.


In [ ]:
from demo_logic.benchmark import representative_selection

selection = representative_selection(benchmark)
show_table(
    ["Split", "Source occurrences", "Selected event types"],
    [
        (name, split_manifest["split_counts"][name], selection["split_counts"][name])
        for name in ("training", "validation", "final_test")
    ],
)
print("Selection version:", selection["version"])
print("Selection fingerprint:", selection["selection_sha256"])
selected_ids = {item["external_id"] for item in selection["records"]}
assert set(benchmark.pilot_ids).issubset(selected_ids)
print("Existing pilot records retained:", len(benchmark.pilot_ids))